In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model and Loss

In [ ]:
"""TransMIT: PyTorch port of the authors' TensorFlow/Keras dual-attention model.

Reference: https://github.com/SJoeQin/TransMIT (TransMIT.py, utils.py, run.py)

RETAINED FROM OFFICIAL RELEASE
- Sliding window of s preceding samples plus one current sample (s=16 by default)
- Independent temporal and feature-axis MHA branches, post-residual LayerNorm
- A *single linear* feedforward Dense(d_model) per block (no invented activation)
- Keras-style MHA with key_dim/value_dim=d_q *per head*
- Temporal and feature-branch projections, concatenation, temporal global-average,
  final Dense(d_in)
- Weighted observed-reconstruction RMSE + missing-imputation RMSE, alpha=0.5
- Autoregressive one-step imputation that feeds earlier imputed estimates forward

ADAPTATIONS FOR AN (B,1080,2) MASKED PHYSIOLOGICAL BENCHMARK
- Torch, explicit observation masks (zero can be a legitimate standardized value)
- Per-fold train-only StandardScaler statistics, set with set_standardization()
- First 16 samples use LEFT ZERO-PADDING rather than remaining unpredicted;
  original utils.online_imputation() starts its outputs at t=s.
- training_loss uses stochastic minibatches of sliding-window examples; this is
  mathematically the official window-training strategy, with optional bounded
  number of sampled windows for GPU memory. By default it teacher-forces
  historical training GT, as the released training code does.
- validation_loss uses held-out autoregressive imputation (no teacher forcing).
  This is a study-specific validation protocol, not the authors' window-validation.
- Compatibility interface: model(x_in,m_hat)->(x_hat,x_c,x_norm_hat,mu,sigma).

NOT OFFICIAL TRAINING CHECKPOINTS: weights are newly trained. Inference requires
one sequential network call for each missing time point that needs imputation.
"""

import math
import torch
from torch import nn
from torch.nn import functional as F


class _KerasStyleMultiHeadAttention(nn.Module):
    """Matches tf.keras.layers.MultiHeadAttention(num_heads, key_dim=d_q, value_dim=d_q)."""
    def __init__(self, embed_dim, num_heads=4, d_q=64, dropout=0.1):
        super().__init__()
        hq = int(num_heads) * int(d_q)
        self.n_heads = int(num_heads)
        self.d_q = int(d_q)
        self.query = nn.Linear(embed_dim, hq)
        self.key = nn.Linear(embed_dim, hq)
        self.value = nn.Linear(embed_dim, hq)
        self.output = nn.Linear(hq, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.query(x).reshape(b, n, self.n_heads, self.d_q).transpose(1, 2)
        k = self.key(x).reshape(b, n, self.n_heads, self.d_q).transpose(1, 2)
        v = self.value(x).reshape(b, n, self.n_heads, self.d_q).transpose(1, 2)
        weights = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_q)
        weights = self.attn_dropout(weights.softmax(dim=-1))
        values = weights @ v
        values = values.transpose(1, 2).contiguous().reshape(b, n, -1)
        return self.output(values)


class _TransMITBlock(nn.Module):
    """Exactly the two residual/LN updates described in official TransMIT.py."""
    def __init__(self, d_model=128, n_heads=4, d_q=64, dropout=0.1):
        super().__init__()
        self.attn = _KerasStyleMultiHeadAttention(d_model, n_heads, d_q, dropout)
        self.ln1 = nn.LayerNorm(d_model, eps=1e-6)
        self.ffn = nn.Linear(d_model, d_model)  # Keras Dense(d_model), default linear
        self.ln2 = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        return self.ln2(x + self.ffn(x))


class TransMITImputer(nn.Module):
    """(B,1080,2) TransMIT port; forward returns standard five-tensor interface.

    Set train-fold standardization statistics before training/inference using:
        model.set_standardization(train_mean, train_std)

    For gradient-based training call training_loss(), *not* forward().
    """
    def __init__(
        self,
        seq_len=1080,
        d_in=2,
        s=16,
        d_model=128,
        d_q=64,
        num_layers=4,
        num_heads=4,
        dropout=0.1,
        alpha=0.5,
        max_train_windows=1024,
    ):
        super().__init__()
        self.seq_len = int(seq_len)
        self.d_in = int(d_in)
        self.s = int(s)
        self.window_len = self.s + 1
        self.alpha = float(alpha)
        self.max_train_windows = max_train_windows
        if self.s < 1 or self.seq_len <= self.s:
            raise ValueError('Need 1 <= s < seq_len')
        if not 0 <= self.alpha <= 1:
            raise ValueError('alpha must be in [0,1]')

        # Official: Dense(d_model) on (B,W,D), plus Dense(d_model) on (B,D,W)
        self.input_temporal = nn.Linear(d_in, d_model)
        self.input_feature = nn.Linear(self.window_len, d_model)
        self.temporal_blocks = nn.ModuleList([
            _TransMITBlock(d_model, num_heads, d_q, dropout)
            for _ in range(num_layers)
        ])
        self.feature_blocks = nn.ModuleList([
            _TransMITBlock(d_model, num_heads, d_q, dropout)
            for _ in range(num_layers)
        ])

        self.temporal_output = nn.Linear(d_model, d_in)
        self.feature_output = nn.Linear(d_model, self.window_len)
        self.fusion_output = nn.Linear(2 * d_in, d_in)
        self.register_buffer('data_mean', torch.zeros(1, 1, d_in))
        self.register_buffer('data_std', torch.ones(1, 1, d_in))
        self.register_buffer('standardization_fitted', torch.tensor(False))

    @torch.no_grad()
    def set_standardization(self, mean, std):
        """Set TRAIN-ONLY channel statistics; never fit using val/test patients."""
        mean = torch.as_tensor(mean, dtype=self.data_mean.dtype, device=self.data_mean.device).reshape(1, 1, self.d_in)
        std = torch.as_tensor(std, dtype=self.data_std.dtype, device=self.data_std.device).reshape(1, 1, self.d_in)
        if not torch.isfinite(mean).all() or not torch.isfinite(std).all() or (std <= 0).any():
            raise ValueError('mean and std must be finite; std strictly positive')
        self.data_mean.copy_(mean)
        self.data_std.copy_(std)
        self.standardization_fitted.fill_(True)

    def predict_window(self, window):
        """Predict current (final-window) value from one sliding window.

        Args: window (N,s+1,D), with missing entries filled with 0 after z-scaling.
        Returns (N,D) predicted standardized channel values.
        """
        if window.ndim != 3 or window.shape[1:] != (self.window_len, self.d_in):
            raise ValueError(f'Expected (N,{self.window_len},{self.d_in}); got {tuple(window.shape)}')
        x = self.input_temporal(window)
        xt = self.input_feature(window.transpose(1, 2))
        for block in self.temporal_blocks:
            x = block(x)
        for block in self.feature_blocks:
            xt = block(xt)
        x = self.temporal_output(x)
        xt = self.feature_output(xt).transpose(1, 2)
        return self.fusion_output(torch.cat((x, xt), dim=-1).mean(dim=1))

    def _check_inputs(self, x_in, m_hat):
        if x_in.ndim != 3 or x_in.shape[1:] != (self.seq_len, self.d_in):
            raise ValueError(f'Expected (B,{self.seq_len},{self.d_in}), got {tuple(x_in.shape)}')
        if m_hat.shape != x_in.shape:
            raise ValueError('m_hat and x_in must have same shape')

    @torch.no_grad()
    def forward(self, x_in, m_hat, predict_observed=False):
        """Sequential inference; optionally predicts even completely observed steps.

        Missing positions are never revealed. The initial s historical positions
        are padded with standardized zeros (training-set mean in raw units).
        predict_observed=True generates predictions at all T time steps,
        matching the authors' inference loop. With False, completely observed
        steps are skipped for faster imputation-only benchmarking.
        """
        self._check_inputs(x_in, m_hat)
        B, T, D = x_in.shape
        mask = (m_hat > 0.5) & torch.isfinite(x_in)
        observed_norm = torch.where(mask, (x_in - self.data_mean) / self.data_std, 0.)
        completed_norm = observed_norm.clone()
        predicted_norm = observed_norm.clone()
        for t in range(T):
            active = (~mask[:, t, :]).any(dim=-1)
            if predict_observed:
                active = torch.ones_like(active)
            if not bool(active.any()):
                continue
            # Independently batch segments that need predictions at time t.
            indices = torch.nonzero(active, as_tuple=False).flatten()
            past = completed_norm[indices, max(0, t-self.s):t, :]
            if past.size(1) < self.s:
                past = F.pad(past, (0, 0, self.s-past.size(1), 0))
            current = observed_norm[indices, t:t+1, :]
            window = torch.cat((past, current), dim=1)
            pred = self.predict_window(window)
            predicted_norm[indices, t, :] = pred
            completed_norm[indices, t, :] = torch.where(
                mask[indices, t, :], observed_norm[indices, t, :], pred
            )

        x_hat = predicted_norm * self.data_std + self.data_mean
        x_c = torch.where(mask, x_in, completed_norm * self.data_std + self.data_mean)
        return x_hat, x_c, predicted_norm, self.data_mean.expand(B, -1, -1), self.data_std.expand(B, -1, -1)

    @staticmethod
    def _weighted_rmse(y_pred, y_true, weight):
        """Official MSE_TransMIT terms: sqrt(sum(weight * error**2)/sum(weight))."""
        n = weight.sum()
        if not bool(n > 0):
            return y_pred.sum() * 0.0
        return torch.sqrt(((y_pred - y_true).square() * weight).sum() / n + 1e-12)

    def training_loss(self, x_in, m_hat, x_true, i_mask, alpha=None, max_train_windows=None):
        """Official-style sliding-window teacher-forced training, weighted RMSE.

        Prior s steps come from TRAINING ground truth, as in TransMIT.py.
        At the current step, only observed x_in values are supplied. All input
        sequences must belong to the current TRAIN fold; never call on test data.
        Stochastic selection from the available sliding windows bounds memory.
        Returns: (total_weighted_rmse, reconstruction_rmse, imputation_rmse).
        """
        self._check_inputs(x_in, m_hat)
        if x_true.shape != x_in.shape or i_mask.shape != x_in.shape:
            raise ValueError('x_true and i_mask must have same shape as x_in')
        a = self.alpha if alpha is None else float(alpha)
        B, T, D = x_in.shape
        valid = torch.isfinite(x_true)
        observed = (m_hat > 0.5) & valid
        targets = (i_mask > 0.5) & (~observed) & valid
        available = observed | targets
        choose = torch.nonzero(available.any(dim=-1).reshape(-1), as_tuple=False).flatten()
        if choose.numel() == 0:
            raise ValueError('No finite observed or artificially withheld ground-truth values')
        max_w = self.max_train_windows if max_train_windows is None else max_train_windows
        if max_w is not None and choose.numel() > max_w:
            choose = choose[torch.randperm(choose.numel(), device=choose.device)[:max_w]]
        # Normalize true history and observed current values separately.
        history_norm = torch.where(valid, (x_true - self.data_mean) / self.data_std, 0.)
        current_norm = torch.where(observed, (x_in - self.data_mean) / self.data_std, 0.)
        # Windows ending at every t: (B,T,W,D); s left-padding for t=0...s-1.
        all_windows = F.pad(history_norm.transpose(1, 2), (self.s, 0)).unfold(-1, self.window_len, 1)
        all_windows = all_windows.permute(0, 2, 3, 1).contiguous().reshape(B*T, self.window_len, D)
        windows = all_windows.index_select(0, choose).clone()
        flat_current = current_norm.reshape(B*T, D)
        windows[:, -1, :] = flat_current.index_select(0, choose)
        y_pred = self.predict_window(windows)
        y_true = history_norm.reshape(B*T, D).index_select(0, choose)
        obs_weight = observed.reshape(B*T, D).float().index_select(0, choose)
        imp_weight = targets.reshape(B*T, D).float().index_select(0, choose)
        recon = self._weighted_rmse(y_pred, y_true, obs_weight)
        impute = self._weighted_rmse(y_pred, y_true, imp_weight)
        total = a * recon + (1-a) * impute
        return total, recon, impute

    @torch.no_grad()
    def validation_loss(self, x_in, m_hat, x_true, i_mask, alpha=None):
        """Actual autoregressive validation RMSE on observed/withheld values.

        Predicting the observed points as well is necessary for the original
        reconstruction RMSE component (x_c would trivially yield zero error).
        """
        predicted, completed, _, _, _ = self.forward(x_in, m_hat, predict_observed=True)
        a = self.alpha if alpha is None else float(alpha)
        valid = torch.isfinite(x_true)
        obs = ((m_hat > 0.5) & valid).float()
        targets = ((i_mask > 0.5) & (m_hat <= 0.5) & valid).float()
        y_gt = torch.where(valid, (x_true - self.data_mean) / self.data_std, 0.)
        y_hat = (predicted - self.data_mean) / self.data_std
        recon = self._weighted_rmse(y_hat, y_gt, obs)
        impute = self._weighted_rmse(y_hat, y_gt, targets)
        total = a*recon + (1-a)*impute
        return total, recon, impute


### Model training, validation, and testing

In [ ]:
NPZ_PATH = (
    '/content/drive/MyDrive/aim3/'
    'saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
)
OUT_DIR = '/content/drive/MyDrive/aim3/TransMIT_1e-4_256'

In [ ]:
"""TransMIT (official-architecture PyTorch port): patient-level five-fold CV.

Run AFTER defining TransMITImputer from:
    transmit_official_style_weighted_rmse_1080x2.py
This script deliberately DOES NOT redefine the model.

Training loss: 0.5 * observed reconstruction RMSE +
               0.5 * artificially missing imputation RMSE.
Training uses the port's sliding-window, teacher-forced training_loss().
Checkpoint selection uses held-out *autoregressive* validation_loss().
Final MAE/RMSE/MAPE use only genuinely withheld artificial target positions.

Adaptations relative to the TensorFlow authors' experiment: mask-based
missingness, train-patient-only StandardScaler, patient CV, left padding
for initial s time steps, and sampled training windows when max_train_windows
is finite. Validation is slow because complete weighted RMSE requires predictions
at every observed time step as well as at the missing time steps.
"""

import os
import random
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Configuration
# ============================================================

CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
SEQ_LEN = 1080
D_IN = 2
BATCH_SIZE = 256
EVAL_BATCH_SIZE = BATCH_SIZE  
GRAD_ACCUM_STEPS = 1
EPOCHS = 300
LR = 1e-4          
EARLY_STOPPING_PATIENCE = 15
VAL_EVERY = 1       
NUM_WORKERS = 0
USE_AMP = False   
CLIP_GRAD_NORM = 1.0
ALPHA = 0.5
MODEL_NAME = 'TransMIT_WeightedRMSE'

MODEL_KWARGS = dict(
    seq_len=SEQ_LEN, d_in=D_IN,
    s=16, d_model=128, d_q=64,
    num_layers=4, num_heads=4, dropout=0.1,
    alpha=ALPHA,
    max_train_windows=1024, 
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16
EPS = 1e-8
REGION_ORDER = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def amp_context():
    return torch.autocast(device_type='cuda', dtype=AMP_DTYPE) if AMP_ENABLED else nullcontext()


class TSImputeDataset(Dataset):
    def __init__(self, x_in, m_hat, x_true, i_mask, seg_indices):
        self.x_in = torch.from_numpy(
            np.nan_to_num(x_in, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.m_hat = torch.from_numpy(np.nan_to_num(m_hat, nan=0.).astype(np.float32))
        # Keep missing GT as NaN; the loss explicitly excludes them.
        self.x_true = torch.from_numpy(x_true.astype(np.float32))
        self.i_mask = torch.from_numpy(np.nan_to_num(i_mask, nan=0.).astype(np.float32))
        self.seg_indices = torch.from_numpy(np.asarray(seg_indices, dtype=np.int64))
        assert self.x_in.shape == self.m_hat.shape == self.x_true.shape == self.i_mask.shape

    def __len__(self):
        return len(self.x_in)

    def __getitem__(self, index):
        return (self.x_in[index], self.m_hat[index], self.x_true[index],
                self.i_mask[index], self.seg_indices[index])


def load_data():
    with np.load(NPZ_PATH, allow_pickle=True) as d:
        x_in, m_hat, x_true, i_mask = (d[k] for k in ('X_in', 'M_hat', 'X', 'I'))
        seg_id = d['segment_ids']
        filenames = (np.asarray([str(v) for v in d['filenames']])
                     if 'filenames' in d.files else
                     np.array([f'segment_{i}' for i in range(len(seg_id))]))
        if 'target_columns' in d.files and len(d['target_columns']) == x_in.shape[-1]:
            target_names = [str(v) for v in d['target_columns']]
        elif 'columns_all' in d.files and len(d['columns_all']) == x_in.shape[-1]:
            target_names = [str(v) for v in d['columns_all']]
        else:
            target_names = [f'channel_{i}' for i in range(x_in.shape[-1])]
    if not (x_in.shape == m_hat.shape == x_true.shape == i_mask.shape):
        raise ValueError('X_in, M_hat, X, I must have identical shapes')
    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f'Expected (N,{SEQ_LEN},{D_IN}), got {x_in.shape}')
    if len(seg_id) != len(x_in):
        raise ValueError('segment_ids length mismatch')
    if np.any((m_hat > 0.5) & (i_mask > 0.5)):
        raise ValueError('M_hat and I overlap: artificial targets must be hidden')
    if not np.isfinite(x_in[m_hat > 0.5]).all():
        raise ValueError('Observed X_in must be finite')
    if not np.isfinite(x_true[i_mask > 0.5]).all():
        raise ValueError('Some artificially hidden targets lack finite GT')
    return x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names


def patient_folds_from_ids(unique_patients):
    pats = np.array(unique_patients).copy()
    np.random.default_rng(SEED).shuffle(pats)
    return np.array_split(pats, N_SPLITS)


def split_train_val_patients(trainval_patients, fold):
    pats = np.array(trainval_patients).copy()
    n_val = min(max(1, int(round(len(pats) * VAL_WITHIN_TRAIN_P))), len(pats) - 1)
    np.random.default_rng(SEED + fold).shuffle(pats)
    return pats[n_val:], pats[:n_val]


def indices_for_patients(seg_id, patient_ids):
    return np.flatnonzero(np.isin(seg_id, patient_ids))


def make_loader(arrays, indices, *, train=False):
    x_in, m_hat, x_true, i_mask = arrays
    ds = TSImputeDataset(x_in[indices], m_hat[indices], x_true[indices],
                         i_mask[indices], indices)
    return DataLoader(ds, batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


def train_epoch(model, loader, optimizer, scaler=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running = np.zeros(3, np.float64)
    n_batches = 0
    for batch_i, (xb, mb, yb, ib, _) in enumerate(loader):
        xb, mb, yb, ib = [v.to(DEVICE, non_blocking=True) for v in (xb, mb, yb, ib)]
        with amp_context():
            total, recon, impute = model.training_loss(xb, mb, yb, ib)
        if not all(torch.isfinite(v).item() for v in (total, recon, impute)):
            raise FloatingPointError('Non-finite TransMIT training loss')
        group_start = (batch_i // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_n = min(GRAD_ACCUM_STEPS, len(loader) - group_start)
        loss = total / group_n
        if scaler is None:
            loss.backward()
        else:
            scaler.scale(loss).backward()
        running += np.array([float(v.detach()) for v in (total, recon, impute)])
        n_batches += 1
        if (batch_i + 1) % GRAD_ACCUM_STEPS == 0 or batch_i + 1 == len(loader):
            if scaler is not None:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            if scaler is None:
                optimizer.step()
            else:
                scaler.step(optimizer)
                scaler.update()
            optimizer.zero_grad(set_to_none=True)
    if not n_batches:
        raise RuntimeError('Empty training loader')
    return tuple(running / n_batches)


@torch.no_grad()
def validate_epoch(model, loader):
    """Pooled normalized RMSE, weighted exactly by observed/target count.

    The per-batch validation_loss uses the original weighted RMSE objective.
    Only x_in and mask are used for autoregressive predictions; GT for scoring.
    """
    model.eval()
    obs_sse = imp_sse = 0.
    obs_count = imp_count = 0
    for xb, mb, yb, ib, _ in loader:
        xb, mb, yb, ib = [v.to(DEVICE, non_blocking=True) for v in (xb, mb, yb, ib)]
        with amp_context():
            _, recon, impute = model.validation_loss(xb, mb, yb, ib)
        if not torch.isfinite(recon).item() or not torch.isfinite(impute).item():
            raise FloatingPointError('Non-finite autoregressive validation RMSE')
        valid = torch.isfinite(yb)
        n_obs = int(((mb > .5) & valid).sum().item())
        n_imp = int(((ib > .5) & (mb <= .5) & valid).sum().item())
        obs_sse += float(recon) ** 2 * n_obs
        imp_sse += float(impute) ** 2 * n_imp
        obs_count += n_obs
        imp_count += n_imp
    if obs_count == 0 or imp_count == 0:
        raise RuntimeError('Validation missing eligible observed or withheld GT points')
    obs_rmse = (obs_sse / obs_count) ** .5
    imp_rmse = (imp_sse / imp_count) ** .5
    return ALPHA * obs_rmse + (1 - ALPHA) * imp_rmse, obs_rmse, imp_rmse


def compute_gap_lengths_from_mask(mask_1d):
    m = np.asarray(mask_1d) > .5
    changes = np.diff(np.r_[False, m, False].astype(np.int8))
    starts, ends = np.flatnonzero(changes == 1), np.flatnonzero(changes == -1)
    result = np.zeros(len(m), dtype=np.int32)
    for start, end in zip(starts, ends):
        result[start:end] = end - start
    return result


def assign_gap_region(g):
    if 1 <= g < 12:
        return '<1 min'
    if 12 <= g < 120:
        return '[1,10) min'
    if 120 <= g < 240:
        return '[10,20) min'
    if 240 <= g <= 360:
        return '[20,30] min'
    return None


@torch.no_grad()
def collect_point_predictions(model, loader, seg_id, filenames, target_names,
                              fold, split='test'):
    """Imputation-only forward skips observed time steps for faster evaluation."""
    model.eval()
    rows = []
    for xb, mb, yb, ib, idxb in loader:
        xb, mb = [v.to(DEVICE, non_blocking=True) for v in (xb, mb)]
        with amp_context():
            _, completed, _, _, _ = model(xb, mb)
        pred_np = completed.float().cpu().numpy()
        truth_np, i_np, obs_np = yb.numpy(), ib.numpy(), mb.cpu().numpy()
        for b, global_idx in enumerate(idxb.tolist()):
            for d, name in enumerate(target_names):
                eligible = (i_np[b, :, d] > .5) & (obs_np[b, :, d] <= .5)
                gap_lengths = compute_gap_lengths_from_mask(eligible)
                for t in np.flatnonzero(eligible):
                    gt, pred = float(truth_np[b, t, d]), float(pred_np[b, t, d])
                    if not np.isfinite(gt) or not np.isfinite(pred):
                        continue
                    g = int(gap_lengths[t])
                    rows.append(dict(
                        fold=int(fold), split=split, segment_idx=int(global_idx),
                        patient_id=str(seg_id[global_idx]),
                        filename=str(filenames[global_idx]), time_idx=int(t),
                        target=str(name), y_true=gt, y_pred=pred,
                        abs_err=abs(gt - pred), gap_len=g,
                        gap_region=assign_gap_region(g), model=MODEL_NAME,
                    ))
    return pd.DataFrame(rows)


def metrics_df(df):
    if df.empty:
        return dict(mae=np.nan, rmse=np.nan, mape_pct=np.nan,
                    n_points=0, n_mape_points=0)
    actual = df['y_true'].to_numpy(dtype=np.float64)
    guess = df['y_pred'].to_numpy(dtype=np.float64)
    delta = np.abs(actual - guess)
    safe = np.abs(actual) > EPS
    return dict(
        mae=float(delta.mean()),
        rmse=float(np.sqrt(np.mean(delta ** 2))),
        mape_pct=float((100 * delta[safe] / np.abs(actual[safe])).mean()) if safe.any() else np.nan,
        n_points=len(df), n_mape_points=int(safe.sum()),
    )


def save_csv(frame, name):
    path = os.path.join(OUT_DIR, name)
    frame.to_csv(path, index=False)
    return path


def _summarize_rows(frame, cols, prefix):
    rec = []
    for keys, group in frame.groupby(cols, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        rec.append({**dict(zip(cols, keys)),
                    **{f'{prefix}_{key}': val for key, val in metrics_df(group).items()}})
    return pd.DataFrame(rec)


def save_summaries(fold_rows, test_rows, val_rows, split_rows):
    fold_df = pd.DataFrame(fold_rows)
    tests = pd.concat(test_rows, ignore_index=True)
    val = pd.concat(val_rows, ignore_index=True)
    save_csv(fold_df, 'cv5_fold_results.csv')
    save_csv(tests, 'cv5_test_predictions_pooled.csv')
    save_csv(val, 'cv5_val_predictions_pooled.csv')
    save_csv(pd.DataFrame(split_rows), 'cv5_patient_split_summary.csv')

    test_gap = tests.dropna(subset=['gap_region']).copy()
    test_gap['gap_region'] = pd.Categorical(test_gap['gap_region'], REGION_ORDER, ordered=True)
    save_csv(test_gap, 'cv5_test_predictions_pooled_with_gap_region.csv')
    save_csv(_summarize_rows(tests, ['target', 'model'], 'pooled_test'),
             'cv5_pooled_test_metrics.csv')
    save_csv(_summarize_rows(val, ['target', 'model'], 'pooled_val'),
             'cv5_pooled_val_metrics.csv')
    save_csv(_summarize_rows(test_gap, ['target', 'model', 'gap_region'], 'pooled_test'),
             'cv5_pooled_test_metrics_by_gap_region.csv')
    per_fold_gap = _summarize_rows(test_gap,
                                  ['fold', 'target', 'model', 'gap_region'], 'test')
    save_csv(per_fold_gap, 'cv5_fold_metrics_by_gap_region.csv')

    fold_summary = fold_df.groupby(['target', 'model'], as_index=False).agg(
        val_weighted_rmse_mean=('val_weighted_rmse', 'mean'),
        val_weighted_rmse_std=('val_weighted_rmse', 'std'),
        val_observed_rmse_mean=('val_observed_rmse', 'mean'),
        val_observed_rmse_std=('val_observed_rmse', 'std'),
        val_imputation_rmse_mean=('val_imputation_rmse', 'mean'),
        val_imputation_rmse_std=('val_imputation_rmse', 'std'),
        val_mae_mean=('val_mae', 'mean'), val_mae_std=('val_mae', 'std'),
        test_mae_mean=('test_mae', 'mean'), test_mae_std=('test_mae', 'std'),
        test_rmse_mean=('test_rmse', 'mean'), test_rmse_std=('test_rmse', 'std'),
        test_mape_mean_pct=('test_mape_pct', 'mean'),
        test_mape_std_pct=('test_mape_pct', 'std'),
        n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'),
    )
    save_csv(fold_summary, 'cv5_summary_mean_std.csv')

    if per_fold_gap.empty:
        gap_summary = pd.DataFrame()
    else:
        gap_summary = per_fold_gap.groupby(
            ['target', 'model', 'gap_region'], observed=True, as_index=False
        ).agg(
            test_mae_mean=('test_mae', 'mean'),
            test_mae_std=('test_mae', 'std'),
            test_rmse_mean=('test_rmse', 'mean'),
            test_rmse_std=('test_rmse', 'std'),
            test_mape_mean_pct=('test_mape_pct', 'mean'),
            test_mape_std_pct=('test_mape_pct', 'std'),
            n_folds=('fold', 'nunique'), total_points=('test_n_points', 'sum'),
        )
    save_csv(gap_summary, 'cv5_summary_mean_std_by_gap_region.csv')
    print('\nCompleted-fold summary:\n', fold_df.to_string(index=False))
    print('\nMean ± SD across completed folds:\n', fold_summary.to_string(index=False))
    print('\nPooled out-of-fold test metrics:\n',
          _summarize_rows(tests, ['target', 'model'], 'pooled_test').to_string(index=False))
    print('\nTest performance by gap region:\n', gap_summary.to_string(index=False))


def main():
    if 'TransMITImputer' not in globals():
        raise NameError(
            'TransMITImputer not found. First run the MODEL definition cell '
            '(transmit_official_style_weighted_rmse_1080x2.py), then run this '
            'script using %run -i /path/to/train_transmit_weighted_rmse_5fold.py'
        )
    if N_SPLITS < 2 or VAL_EVERY < 1 or GRAD_ACCUM_STEPS < 1:
        raise ValueError('Invalid cross-validation/training configuration')
    seed_all(SEED)
    os.makedirs(CKPT_DIR, exist_ok=True)
    x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names = load_data()
    arrays = (x_in, m_hat, x_true, i_mask)
    unique_patients = np.unique(seg_id)
    if len(unique_patients) < N_SPLITS + 2:
        raise ValueError('Not enough distinct patient IDs for CV and validation')
    patient_folds = patient_folds_from_ids(unique_patients)
    print(f'DEVICE={DEVICE}, AMP={AMP_ENABLED}, segments={len(seg_id)}, '
          f'patients={len(unique_patients)}, targets={target_names}')
    print(f'Model: {MODEL_KWARGS} | batch={BATCH_SIZE}, eval_batch={EVAL_BATCH_SIZE}')
    print(f'Loss: {ALPHA} × observed RMSE + {1-ALPHA} × withheld RMSE '
          '(both in training-standardized units)')
    print('WARNING: Validation predicts at every time step and may be slow.')

    fold_rows, test_rows, val_rows, split_rows = [], [], [], []
    for fold, test_patients in enumerate(patient_folds, start=1):
        excluded = set(test_patients.tolist())
        trainval_patients = np.asarray([p for p in unique_patients if p not in excluded])
        train_patients, val_patients = split_train_val_patients(trainval_patients, fold)
        train_idx = indices_for_patients(seg_id, train_patients)
        val_idx = indices_for_patients(seg_id, val_patients)
        test_idx = indices_for_patients(seg_id, test_patients)
        if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
            raise ValueError(f'Fold {fold} has an empty train/val/test partition')
        print('\n' + '=' * 80)
        print(f'FOLD {fold}/{N_SPLITS}: patients '
              f'{len(train_patients)}/{len(val_patients)}/{len(test_patients)}, '
              f'segments {len(train_idx)}/{len(val_idx)}/{len(test_idx)}')
        print('=' * 80)
        split_rows.append(dict(
            fold=fold,
            train_patients=','.join(map(str, sorted(train_patients.tolist()))),
            val_patients=','.join(map(str, sorted(val_patients.tolist()))),
            test_patients=','.join(map(str, sorted(test_patients.tolist()))),
            n_train_patients=len(train_patients), n_val_patients=len(val_patients),
            n_test_patients=len(test_patients),
            n_train_segments=len(train_idx), n_val_segments=len(val_idx),
            n_test_segments=len(test_idx),
        ))
        train_loader = make_loader(arrays, train_idx, train=True)
        val_loader = make_loader(arrays, val_idx)
        test_loader = make_loader(arrays, test_idx)

        model = TransMITImputer(**MODEL_KWARGS).to(DEVICE)
        # Train-fold statistics include artificial GT values only from TRAIN patients.
        # Non-finite naturally unobserved GT entries are excluded.
        train_truth = np.asarray(x_true[train_idx], dtype=np.float64)
        valid = np.isfinite(train_truth)
        count = valid.sum(axis=(0, 1))
        if np.any(count == 0):
            raise ValueError(f'Fold {fold}: no finite training GT in a channel')
        mean = np.where(valid, train_truth, 0.).sum(axis=(0, 1)) / count
        ss = np.where(valid, (train_truth - mean) ** 2, 0.).sum(axis=(0, 1))
        std = np.sqrt(ss / count).clip(min=1e-6)
        model.set_standardization(mean, std)
        print(f'Train-only standardization mean={mean}, std={std}')
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        scaler = (torch.amp.GradScaler('cuda') if AMP_ENABLED and
                  AMP_DTYPE == torch.float16 else None)
        best_val, best_epoch, bad_epochs = float('inf'), 0, 0
        checkpoint = os.path.join(CKPT_DIR, f'fold{fold:02d}_best.pt')
        history = []
        for epoch in range(1, EPOCHS + 1):
            tr_total, tr_recon, tr_imp = train_epoch(model, train_loader, optimizer, scaler)
            if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
                va_total, va_recon, va_imp = validate_epoch(model, val_loader)
                print(f'Epoch {epoch:03d} | LR={LR:.2e} | '
                      f'train weighted RMSE={tr_total:.4f} '
                      f'(obs={tr_recon:.4f}, gap={tr_imp:.4f}) | '
                      f'val weighted RMSE={va_total:.4f} '
                      f'(obs={va_recon:.4f}, gap={va_imp:.4f})')
                if va_total < best_val:
                    best_val, best_epoch, bad_epochs = va_total, epoch, 0
                    torch.save(model.state_dict(), checkpoint)
                    print('  Saved best checkpoint:', checkpoint)
                else:
                    bad_epochs += 1
            else:
                va_total = va_recon = va_imp = np.nan
            history.append(dict(
                fold=fold, epoch=epoch, lr=LR,
                train_weighted_rmse=tr_total,
                train_observed_rmse=tr_recon,
                train_imputation_rmse=tr_imp,
                val_weighted_rmse=va_total,
                val_observed_rmse=va_recon,
                val_imputation_rmse=va_imp,
            ))
            save_csv(pd.DataFrame(history), f'fold{fold:02d}_training_history.csv')
            if bad_epochs >= EARLY_STOPPING_PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break
        if best_epoch == 0:
            raise RuntimeError(f'Fold {fold} did not save a valid checkpoint')
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE,
                                         weights_only=True))
        model.eval()
        print(f'Fold {fold}: best epoch={best_epoch}, val weighted RMSE={best_val:.4f}')
        validation_predictions = collect_point_predictions(
            model, val_loader, seg_id, filenames, target_names, fold, split='val')
        test_predictions = collect_point_predictions(
            model, test_loader, seg_id, filenames, target_names, fold, split='test')
        if test_predictions.empty or validation_predictions.empty:
            raise RuntimeError(f'Fold {fold} has no finite withheld predictions')
        save_csv(test_predictions, f'fold{fold:02d}_test_predictions.csv')
        save_csv(validation_predictions, f'fold{fold:02d}_val_predictions.csv')
        test_rows.append(test_predictions)
        val_rows.append(validation_predictions)

        best_history = next(h for h in history if h['epoch'] == best_epoch)
        for target, subset in test_predictions.groupby('target'):
            scores = metrics_df(subset)
            vscores = metrics_df(validation_predictions[
                validation_predictions.target == target])
            fold_rows.append(dict(
                fold=fold, target=target, model=MODEL_NAME,
                val_weighted_rmse=best_val,
                val_observed_rmse=best_history['val_observed_rmse'],
                val_imputation_rmse=best_history['val_imputation_rmse'],
                val_mae=vscores['mae'], val_rmse=vscores['rmse'],
                val_mape_pct=vscores['mape_pct'],
                test_mae=scores['mae'], test_rmse=scores['rmse'],
                test_mape_pct=scores['mape_pct'], n_points=scores['n_points'],
                n_train_patients=len(train_patients), n_val_patients=len(val_patients),
                n_test_patients=len(test_patients),
                n_train_segments=len(train_idx), n_val_segments=len(val_idx),
                n_test_segments=len(test_idx), best_epoch=best_epoch,
            ))
            print(f'  {target} | VAL MAE={vscores["mae"]:.3f} | '
                  f'TEST MAE={scores["mae"]:.3f} RMSE={scores["rmse"]:.3f} '
                  f'MAPE={scores["mape_pct"]:.2f}%')
        save_summaries(fold_rows, test_rows, val_rows, split_rows)
        del model, optimizer
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    print('\nFive-fold patient-level CV complete. Results:', OUT_DIR)

if __name__ == '__main__':
    main()

DEVICE=cuda, AMP=False, segments=3817, patients=64, targets=['channel_0', 'channel_1']
Model: {'seq_len': 1080, 'd_in': 2, 's': 16, 'd_model': 128, 'd_q': 64, 'num_layers': 4, 'num_heads': 4, 'dropout': 0.1, 'alpha': 0.5, 'max_train_windows': 1024} | batch=256, eval_batch=256
Loss: 0.5 × observed RMSE + 0.5 × withheld RMSE (both in training-standardized units)

FOLD 1/5: patients 46/5/13, segments 2814/256/747
Train-only standardization mean=[61.83005749 59.10847674], std=[12.89989791 13.88566159]
Epoch 001 | LR=1.00e-04 | train weighted RMSE=0.8200 (obs=0.8405, gap=0.7996) | val weighted RMSE=1.0441 (obs=0.8504, gap=1.2378)
  Saved best checkpoint: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/ckpts/fold01_best.pt
Epoch 002 | LR=1.00e-04 | train weighted RMSE=0.6324 (obs=0.6369, gap=0.6279) | val weighted RMSE=0.9571 (obs=0.7555, gap=1.1588)
  Saved best checkpoint: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/ckpts/fold01_best.pt
Epoch 003 | LR=1.00e-04 | train weighted RMSE=0.5711 

### Validation performance

In [ ]:

import os
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset


SPLIT_CSV = os.path.join(OUT_DIR, 'cv5_patient_split_summary.csv')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')

SEQ_LEN = 1080
D_IN = 2
N_SPLITS = 5
EVAL_BATCH_SIZE = 128 
NUM_WORKERS = 0
USE_AMP = False 
ALPHA = 0.5
EPS = 1e-8
MODEL_NAME = 'TransMIT_WeightedRMSE'
REGION_ORDER = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']

MODEL_KWARGS = dict(
    seq_len=SEQ_LEN, d_in=D_IN, s=16, d_model=128, d_q=64,
    num_layers=4, num_heads=4, dropout=0.1, alpha=ALPHA,
    max_train_windows=1024,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'
AMP_DTYPE = (torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported()
             else torch.float16)
PIN_MEMORY = DEVICE.type == 'cuda'


def amp_context():
    if AMP_ENABLED:
        return torch.autocast(device_type='cuda', dtype=AMP_DTYPE)
    return nullcontext()


def save_csv(frame, name):
    path = os.path.join(OUT_DIR, name)
    frame.to_csv(path, index=False)
    print('Saved:', path)
    return path


def parse_patient_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [s.strip() for s in str(value).split(',') if s.strip()]


def normalize_patient_id(value):
    """Matches integer, float-formatted or textual IDs without guessing IDs."""
    value = str(value).strip()
    try:
        f = float(value)
        if np.isfinite(f) and f.is_integer():
            return str(int(f))
    except (ValueError, OverflowError):
        pass
    return value


class ValidationDataset(Dataset):
    def __init__(self, x_in, m_hat, x_true, i_mask, indices):
        self.x_in = torch.from_numpy(np.nan_to_num(
            x_in, nan=0., posinf=0., neginf=0.).astype('float32'))
        self.m_hat = torch.from_numpy(np.nan_to_num(m_hat, nan=0.).astype('float32'))
        self.x_true = torch.from_numpy(x_true.astype('float32'))  # Preserve NaN GT.
        self.i_mask = torch.from_numpy(np.nan_to_num(i_mask, nan=0.).astype('float32'))
        self.indices = torch.from_numpy(np.asarray(indices, dtype=np.int64))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return (self.x_in[i], self.m_hat[i], self.x_true[i],
                self.i_mask[i], self.indices[i])


def load_npz():
    if not os.path.isfile(NPZ_PATH):
        raise FileNotFoundError(f'NPZ dataset not found: {NPZ_PATH}')
    with np.load(NPZ_PATH, allow_pickle=True) as d:
        x_in, m_hat, x_true, i_mask = [d[k] for k in ('X_in', 'M_hat', 'X', 'I')]
        patient_ids = d['segment_ids']
        filenames = (np.array([str(s) for s in d['filenames']])
                     if 'filenames' in d.files else
                     np.array([f'segment_{i}' for i in range(len(x_in))]))
        if 'target_columns' in d.files and len(d['target_columns']) == D_IN:
            target_names = list(map(str, d['target_columns']))
        elif 'columns_all' in d.files and len(d['columns_all']) == D_IN:
            target_names = list(map(str, d['columns_all']))
        else:
            target_names = [f'channel_{i}' for i in range(D_IN)]
    if not (x_in.shape == m_hat.shape == x_true.shape == i_mask.shape):
        raise ValueError('X_in/M_hat/X/I shapes differ')
    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f'Expected (*,{SEQ_LEN},{D_IN}), got {x_in.shape}')
    if len(patient_ids) != len(x_in):
        raise ValueError('Patient ID count does not match segment count')
    if np.any((m_hat > .5) & (i_mask > .5)):
        raise ValueError('M_hat and I masks overlap: target leakage')
    if not np.isfinite(x_in[m_hat > .5]).all():
        raise ValueError('X_in has non-finite observed values')
    if not np.isfinite(x_true[(i_mask > .5) & (m_hat <= .5)]).all():
        raise ValueError('Artificial evaluation targets contain non-finite ground truth')
    return x_in, m_hat, x_true, i_mask, patient_ids, filenames, target_names


def gap_lengths(target_mask):
    mask = np.asarray(target_mask) > 0.5
    changes = np.diff(np.r_[False, mask, False].astype(np.int8))
    starts, ends = np.flatnonzero(changes == 1), np.flatnonzero(changes == -1)
    out = np.zeros(len(mask), dtype=np.int32)
    for start, end in zip(starts, ends):
        out[start:end] = end-start
    return out


def gap_region(n):
    if 1 <= n < 12:
        return '<1 min'
    if 12 <= n < 120:
        return '[1,10) min'
    if 120 <= n < 240:
        return '[10,20) min'
    if 240 <= n <= 360:
        return '[20,30] min'
    return None


def metrics(frame):
    if frame.empty:
        return {'mae': np.nan, 'rmse': np.nan, 'mape_pct': np.nan,
                'n_points': 0, 'n_mape_points': 0}
    y = frame['y_true'].to_numpy(dtype=np.float64)
    p = frame['y_pred'].to_numpy(dtype=np.float64)
    finite = np.isfinite(y) & np.isfinite(p)
    d = np.abs(y[finite]-p[finite])
    mape = finite & (np.abs(y) > EPS)
    return dict(
        mae=float(d.mean()) if len(d) else np.nan,
        rmse=float(np.sqrt((d*d).mean())) if len(d) else np.nan,
        mape_pct=float((100*np.abs(y[mape]-p[mape])/np.abs(y[mape])).mean())
        if mape.any() else np.nan,
        n_points=int(finite.sum()), n_mape_points=int(mape.sum()),
    )


def grouped_metrics(frame, columns):
    return pd.DataFrame([
        {**dict(zip(columns, key if isinstance(key, tuple) else (key,))),
         **metrics(group)}
        for key, group in frame.groupby(columns, observed=True)
    ])


def summarize_folds(frame, group_columns):
    if frame.empty:
        return pd.DataFrame()
    return frame.groupby(group_columns, observed=True, as_index=False).agg(
        mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
        rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
        mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
        n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'),
    )


@torch.no_grad()
def evaluate_fold(model, loader, fold, patient_ids, filenames, target_names):
    """One autoregressive pass per batch; compute pooled weighted RMSE and raw MAE.

    The original TransMIT RMSE loss is calculated on RAW PREDICTIONS (including
    predicted values at observed positions), in per-fold standardized units.
    The MAE/RMSE/MAPE metrics are only on withheld targets and in raw units.
    """
    model.eval()
    obs_sse = imp_sse = 0.
    obs_count = imp_count = 0
    rows = []
    for xb, mb, yb, ib, idxb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        mb = mb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        ib = ib.to(DEVICE, non_blocking=True)
        with amp_context():
            predicted, completed, _, _, _ = model(xb, mb, predict_observed=True)
        valid = torch.isfinite(yb)
        observed = (mb > .5) & valid
        withheld = (ib > .5) & (mb <= .5) & valid
        error_standardized = (
            (predicted.float() - torch.where(valid, yb, 0.).float())
            / model.data_std.float()
        )
        obs_sse += (error_standardized.square() * observed).sum().item()
        imp_sse += (error_standardized.square() * withheld).sum().item()
        obs_count += int(observed.sum().item())
        imp_count += int(withheld.sum().item())

        y_np = yb.cpu().numpy()
        p_np = completed.float().cpu().numpy()
        i_np = ib.cpu().numpy()
        m_np = mb.cpu().numpy()
        for b, global_index in enumerate(idxb.tolist()):
            for channel, target_name in enumerate(target_names):
                eligible = (i_np[b,:,channel] > .5) & (m_np[b,:,channel] <= .5)
                lengths = gap_lengths(eligible)
                for t in np.flatnonzero(eligible):
                    gt = float(y_np[b,t,channel])
                    pred = float(p_np[b,t,channel])
                    if not (np.isfinite(gt) and np.isfinite(pred)):
                        continue
                    duration = int(lengths[t])
                    rows.append(dict(
                        fold=int(fold), split='val', segment_idx=int(global_index),
                        patient_id=str(patient_ids[global_index]),
                        filename=str(filenames[global_index]),
                        time_idx=int(t), target=str(target_name),
                        y_true=gt, y_pred=pred, abs_err=abs(gt-pred),
                        gap_len=duration, gap_region=gap_region(duration),
                        model=MODEL_NAME,
                    ))
    if obs_count == 0 or imp_count == 0:
        raise ValueError(f'Fold {fold}: no eligible observed or withheld values')
    if not np.isfinite([obs_sse, imp_sse]).all():
        raise FloatingPointError(f'Fold {fold}: non-finite squared errors')
    rec_rmse = float(np.sqrt(obs_sse / obs_count))
    imp_rmse = float(np.sqrt(imp_sse / imp_count))
    total = float(ALPHA * rec_rmse + (1-ALPHA) * imp_rmse)
    return (total, rec_rmse, imp_rmse, obs_count, imp_count,
            pd.DataFrame(rows))


def main():
    if 'TransMITImputer' not in globals():
        raise NameError(
            'TransMITImputer not defined. Run your TransMIT architecture cell first; '
            'when executing this file in Colab use %run -i so the class is visible.'
        )
    if not os.path.isfile(SPLIT_CSV):
        raise FileNotFoundError(
            f'No patient split CSV at {SPLIT_CSV}. First train TransMIT '
            'with train_transmit_weighted_rmse_5fold.py; do NOT reuse C2FFormer checkpoints.'
        )
    os.makedirs(OUT_DIR, exist_ok=True)
    x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names = load_npz()
    splits = pd.read_csv(SPLIT_CSV).sort_values('fold')
    if splits.empty or splits['fold'].nunique() != N_SPLITS:
        raise ValueError(f'Expected {N_SPLITS} completed folds in {SPLIT_CSV}')
    patient_strings = np.array([normalize_patient_id(v) for v in seg_id])
    losses, predictions = [], []
    print(f'Device: {DEVICE} | Segments: {len(x_in)} | Patients: '
          f'{len(set(patient_strings))} | Channels: {target_names}')
    print(f'Loss: {ALPHA} * observed RMSE + {1-ALPHA} * imputed RMSE, '
          'in TRAIN-FOLD standardized units')
    print('Autoregressive validation (1080 steps per segment) may be slow.')

    for _, row in splits.iterrows():
        fold = int(row['fold'])
        val_pats = {normalize_patient_id(p) for p in parse_patient_ids(row['val_patients'])}
        indices = np.flatnonzero(np.isin(patient_strings, list(val_pats)))
        if not len(indices):
            raise ValueError(f'Fold {fold}: no validation segments match saved IDs')
        if 'n_val_segments' in row and pd.notna(row['n_val_segments']):
            if len(indices) != int(row['n_val_segments']):
                raise ValueError(f'Fold {fold}: validation segment count differs '
                                 'from saved patient split: check input dataset')
        checkpoint = os.path.join(CKPT_DIR, f'fold{fold:02d}_best.pt')
        if not os.path.isfile(checkpoint):
            raise FileNotFoundError(f'Missing TransMIT checkpoint: {checkpoint}')
        model = TransMITImputer(**MODEL_KWARGS).to(DEVICE)
        weights = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
        model.load_state_dict(weights, strict=True)
        if not model.standardization_fitted.item():
            raise RuntimeError(f'Fold {fold}: checkpoint has no fitted '
                               'TRAIN-only standardization statistics')
        model.eval()
        dataset = ValidationDataset(x_in[indices], m_hat[indices], x_true[indices],
                                    i_mask[indices], indices)
        loader = DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        total, reconstruction, imputation, n_obs, n_imp, fold_pred = evaluate_fold(
            model, loader, fold, seg_id, filenames, target_names
        )
        if fold_pred.empty:
            raise ValueError(f'Fold {fold}: no valid validation imputation predictions')
        loss_info = dict(
            fold=fold, model=MODEL_NAME,
            val_weighted_rmse=total,
            val_observed_rmse=reconstruction,
            val_imputation_rmse=imputation,
            n_val_patients=len(val_pats), n_val_segments=len(indices),
            n_observed_points=n_obs, n_target_points=n_imp,
        )
        losses.append(loss_info)
        predictions.append(fold_pred)
        save_csv(fold_pred, f'fold{fold:02d}_validation_predictions_recomputed.csv')
        save_csv(pd.DataFrame(losses), 'cv5_validation_losses_recomputed.csv')
        print(f'Fold {fold}/{N_SPLITS} | segments={len(indices)} | '
              f'weighted RMSE={total:.5f} (observed={reconstruction:.5f}, '
              f'imputation={imputation:.5f})')
        for target_name, sub in fold_pred.groupby('target'):
            score = metrics(sub)
            print(f'  {target_name}: MAE={score["mae"]:.5f} '
                  f'RMSE={score["rmse"]:.5f} MAPE={score["mape_pct"]:.3f}%')
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    loss_df = pd.DataFrame(losses).sort_values('fold')
    loss_summary = pd.DataFrame([dict(
        model=MODEL_NAME,
        val_weighted_rmse_mean=loss_df['val_weighted_rmse'].mean(),
        val_weighted_rmse_std=loss_df['val_weighted_rmse'].std(ddof=1),
        val_observed_rmse_mean=loss_df['val_observed_rmse'].mean(),
        val_observed_rmse_std=loss_df['val_observed_rmse'].std(ddof=1),
        val_imputation_rmse_mean=loss_df['val_imputation_rmse'].mean(),
        val_imputation_rmse_std=loss_df['val_imputation_rmse'].std(ddof=1),
        n_folds=loss_df['fold'].nunique(),
    )])
    save_csv(loss_summary, 'cv5_validation_loss_summary_mean_std.csv')
    all_preds = pd.concat(predictions, ignore_index=True)
    save_csv(all_preds, 'cv5_validation_predictions_pooled_recomputed.csv')
    valid_gaps = all_preds.dropna(subset=['gap_region']).copy()
    valid_gaps['gap_region'] = pd.Categorical(
        valid_gaps['gap_region'], categories=REGION_ORDER, ordered=True)
    save_csv(valid_gaps, 'cv5_validation_predictions_pooled_by_gap_region.csv')
    fold_metrics = grouped_metrics(all_preds, ['fold', 'target', 'model'])
    save_csv(fold_metrics, 'cv5_validation_fold_metrics.csv')
    target_summary = summarize_folds(fold_metrics, ['target', 'model'])
    save_csv(target_summary, 'cv5_validation_metrics_mean_std.csv')
    save_csv(grouped_metrics(all_preds, ['target', 'model']),
             'cv5_validation_pooled_metrics.csv')
    gap_fold_metrics = grouped_metrics(
        valid_gaps, ['fold', 'target', 'model', 'gap_region'])
    save_csv(gap_fold_metrics, 'cv5_validation_fold_metrics_by_gap_region.csv')
    save_csv(summarize_folds(gap_fold_metrics, ['target', 'model', 'gap_region']),
             'cv5_validation_metrics_mean_std_by_gap_region.csv')
    save_csv(grouped_metrics(valid_gaps, ['target', 'model', 'gap_region']),
             'cv5_validation_pooled_metrics_by_gap_region.csv')
    print('\nFive-fold recomputed validation weighted RMSE (mean ± sample SD):')
    print(loss_summary.to_string(index=False))
    print('\nValidation MAE/RMSE/MAPE by target (mean ± sample SD):')
    print(target_summary.to_string(index=False))
    print('\nSaved validation results to', OUT_DIR)


if __name__ == '__main__':
    main()

Device: cuda | Segments: 3817 | Patients: 64 | Channels: ['channel_0', 'channel_1']
Loss: 0.5 * observed RMSE + 0.5 * imputed RMSE, in TRAIN-FOLD standardized units
Autoregressive validation (1080 steps per segment) may be slow.
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/fold01_validation_predictions_recomputed.csv
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/cv5_validation_losses_recomputed.csv
Fold 1/5 | segments=256 | weighted RMSE=0.40653 (observed=0.37152, imputation=0.44155)
  channel_0: MAE=3.99156 RMSE=5.67286 MAPE=6.044%
  channel_1: MAE=4.54126 RMSE=6.21779 MAPE=7.258%
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/fold02_validation_predictions_recomputed.csv
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/cv5_validation_losses_recomputed.csv
Fold 2/5 | segments=303 | weighted RMSE=0.28695 (observed=0.22714, imputation=0.34676)
  channel_0: MAE=3.24034 RMSE=4.86126 MAPE=4.742%
  channel_1: MAE=1.75027 RMSE=3.02500 MAPE=3.026%
Saved: /content/drive

### Model Testing

In [ ]:
import os
import re
import numpy as np
import pandas as pd

PRED_CSV = os.path.join(OUT_DIR, 'cv5_test_predictions_pooled.csv')
MODEL_NAME = 'TransMIT_WeightedRMSE'
EPS = 1e-8
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']


def canonical_target(label):
    """Convert training-script target_names to stated channel names safely.

    The source training script saves NAME rather than CHANNEL INDEX in its CSV.
    Therefore the saved label must identify the channel; unknown labels are
    rejected instead of silently assigning ScO2/MAP in the wrong order.
    """
    name = str(label).strip()
    compact = re.sub(r'[^a-z0-9]', '', name.lower().replace('₂', '2'))
    if (compact in ('0', 'channel0', 'target0', 'sco2', 'rso2', 'cerebralso2')
            or 'sco2' in compact or 'cerebraloxygensaturation' in compact):
        return 'ScO2'
    if (compact in ('1', 'channel1', 'target1', 'map', 'mapmmhg')
            or 'meanarterialpressure' in compact):
        return 'MAP'
    raise ValueError(
        f'Unrecognized target label {name!r}. The prediction CSV stores a target '
        'name, not channel index. Add the correct name to canonical_target() '
        'only after verifying target_columns in the source NPZ '
        '(channel 0=ScO2; channel 1=MAP).'
    )


def calculate_metrics(frame):
    yt = frame['y_true'].to_numpy(dtype=np.float64)
    yp = frame['y_pred'].to_numpy(dtype=np.float64)
    if len(yt) == 0:
        return dict(mae=np.nan, mape_pct=np.nan,
                    n_points_total=0, n_points_used_mape=0,
                    n_points_excluded_gt0=0)
    ae = np.abs(yp - yt)
    nonzero = np.abs(yt) > EPS
    return dict(
        mae=float(np.mean(ae)),
        mape_pct=(float(100 * np.mean(ae[nonzero] / np.abs(yt[nonzero])))
                  if nonzero.any() else np.nan),
        n_points_total=int(len(yt)),
        n_points_used_mape=int(nonzero.sum()),
        n_points_excluded_gt0=int((~nonzero).sum()),
    )


def mean_sd(mean, std):
    if pd.isna(mean):
        return 'NA'
    if pd.isna(std):
        return f'{mean:.2f} ± NA'
    return f'{mean:.2f} ± {std:.2f}'


def main():
    if not os.path.isfile(PRED_CSV):
        raise FileNotFoundError(
            f'Cannot find: {PRED_CSV}\n'
            'First run the TransMIT five-fold training/evaluation script. '
            'This reporting script reads predictions only.'
        )

    pred_df = pd.read_csv(PRED_CSV)
    required = {'fold', 'target', 'y_true', 'y_pred', 'gap_len'}
    absent = required - set(pred_df.columns)
    if absent:
        raise ValueError(f'Missing columns from {PRED_CSV}: {sorted(absent)}')
    if 'split' in pred_df.columns:
        unexpected = set(pred_df['split'].dropna().astype(str).str.lower()) - {'test'}
        if unexpected:
            raise ValueError(f'Non-test rows in {PRED_CSV}: {sorted(unexpected)}')

    if 'model' not in pred_df.columns:
        pred_df['model'] = MODEL_NAME
    unexpected_models = set(pred_df['model'].dropna().unique()) - {MODEL_NAME}
    if unexpected_models:
        raise ValueError(
            f'Unexpected model labels {unexpected_models}; expected {MODEL_NAME!r}. '
            'Check that OUT_DIR points to the TransMIT experiment.'
        )

    for col in ('fold', 'y_true', 'y_pred', 'gap_len'):
        pred_df[col] = pd.to_numeric(pred_df[col], errors='coerce')
    bad_folds = pred_df['fold'].isna() | (pred_df['fold'] % 1 != 0)
    if bad_folds.any():
        raise ValueError(f'Invalid fold identifiers in {int(bad_folds.sum())} rows')
    pred_df['fold'] = pred_df['fold'].astype(int)
    n_bad = (~(np.isfinite(pred_df['y_true']) & np.isfinite(pred_df['y_pred']))).sum()
    if n_bad:
        print(f'Excluding {n_bad:,} rows with nonfinite truth or prediction.')
        pred_df = pred_df.loc[
            np.isfinite(pred_df['y_true']) & np.isfinite(pred_df['y_pred'])
        ].copy()
    if pred_df.empty:
        raise ValueError('No finite held-out test predictions found')

    print('Raw channel names in predictions:', pred_df['target'].dropna().unique().tolist())
    pred_df['target'] = pred_df['target'].map(canonical_target)
    print('Standardized channel labels:', pred_df['target'].unique().tolist())
    available_folds = sorted(pred_df['fold'].unique().tolist())
    if len(available_folds) != 5:
        print(f'WARNING: found {len(available_folds)} test folds {available_folds}; '
              'results are computed over available folds only.')

    dedupe_cols = ['fold', 'segment_idx', 'time_idx', 'target']
    if set(dedupe_cols).issubset(pred_df.columns):
        dup = pred_df.duplicated(dedupe_cols)
        if dup.any():
            raise ValueError(f'{int(dup.sum())} duplicated held-out test rows; '
                             'check cv5_test_predictions_pooled.csv')

    pred_df['gap_region'] = pd.cut(
        pred_df['gap_len'], bins=[0, 11, 119, 239, 360],
        labels=REGIONS[:-1], include_lowest=True,
    )
    n_outside = int(pred_df['gap_region'].isna().sum())
    if n_outside:
        print(f'WARNING: {n_outside:,} points have gap_len outside 1..360 '
              'or missing; retained in ALL but omitted from four duration bins.')

    print('\nTest points by gap category:')
    print(pred_df['gap_region'].value_counts(sort=False).to_string())
    print(f'ALL test points: {len(pred_df):,}')

    gap_points = pred_df.loc[pred_df['gap_region'].notna()].copy()
    all_points = pred_df.copy()
    all_points['gap_region'] = 'ALL'
    eval_df = pd.concat([gap_points, all_points], ignore_index=True)
    eval_df['gap_region'] = pd.Categorical(
        eval_df['gap_region'], categories=REGIONS, ordered=True
    )

    fold_rows = []
    for (fold, target, model, region), part in eval_df.groupby(
        ['fold', 'target', 'model', 'gap_region'], observed=True
    ):
        fold_rows.append(dict(
            fold=int(fold), target=target, model=model,
            gap_region=str(region), **calculate_metrics(part)
        ))
    fold_df = pd.DataFrame(fold_rows)
    fold_df['gap_region'] = pd.Categorical(fold_df['gap_region'], REGIONS, ordered=True)
    fold_df = fold_df.sort_values(
        ['target', 'model', 'gap_region', 'fold']
    ).reset_index(drop=True)

    summary_df = (fold_df.groupby(
        ['target', 'model', 'gap_region'], observed=True
    ).agg(
        mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
        mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
        n_folds_mae=('mae', 'count'), n_folds_mape=('mape_pct', 'count'),
        total_points=('n_points_total', 'sum'),
        total_points_used_mape=('n_points_used_mape', 'sum'),
        total_points_excluded_gt0=('n_points_excluded_gt0', 'sum'),
    ).reset_index().sort_values(
        ['target', 'model', 'gap_region']
    ).reset_index(drop=True))

    pooled_rows = []
    for (target, model, region), part in eval_df.groupby(
        ['target', 'model', 'gap_region'], observed=True
    ):
        pooled_rows.append(dict(
            target=target, model=model, gap_region=str(region),
            **calculate_metrics(part)
        ))
    pooled_df = pd.DataFrame(pooled_rows)
    pooled_df['gap_region'] = pd.Categorical(pooled_df['gap_region'], REGIONS, ordered=True)
    pooled_df = pooled_df.sort_values(
        ['target', 'model', 'gap_region']
    ).reset_index(drop=True)

    paper_rows = []
    for (target, model), part in summary_df.groupby(['target', 'model'], observed=True):
        row = {'target': target, 'model': model}
        for region in REGIONS:
            match = part.loc[part['gap_region'] == region]
            if match.empty:
                row[f'{region} | MAE'] = 'NA'
                row[f'{region} | MAPE(%)'] = 'NA'
            else:
                r = match.iloc[0]
                row[f'{region} | MAE'] = mean_sd(r['mae_mean'], r['mae_std'])
                row[f'{region} | MAPE(%)'] = mean_sd(r['mape_mean_pct'], r['mape_std_pct'])
        paper_rows.append(row)
    paper_df = pd.DataFrame(paper_rows)
    paper_df['target'] = pd.Categorical(paper_df['target'], ['ScO2', 'MAP'], ordered=True)
    paper_df = paper_df.sort_values('target').reset_index(drop=True)
    paper_df['target'] = paper_df['target'].astype(str)

    outputs = {
        'cv5_fold_mae_mape_by_gap_region.csv': fold_df,
        'cv5_summary_mean_std_mae_mape_by_gap_region.csv': summary_df,
        'cv5_pooled_test_mae_mape_by_gap_region.csv': pooled_df,
        'transmit_paper4_mae_mape_table.csv': paper_df,
    }
    os.makedirs(OUT_DIR, exist_ok=True)
    for filename, frame in outputs.items():
        path = os.path.join(OUT_DIR, filename)
        frame.to_csv(path, index=False)
        print('Saved:', path)

    print('\nFOLDWISE TEST MAE + MAPE:\n', fold_df.to_string(index=False))
    print('\nFIVE-FOLD MEAN ± SAMPLE SD:\n', summary_df.to_string(index=False))
    print('\nMANUSCRIPT-READY TABLE:\n', paper_df.to_string(index=False))


if __name__ == '__main__':
    main()


Raw channel names in predictions: ['channel_0', 'channel_1']
Standardized channel labels: ['ScO2', 'MAP']

Test points by gap category:
gap_region
<1 min         39503
[1,10) min     52673
[10,20) min    26458
[20,30] min    25794
ALL test points: 144,428
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/cv5_fold_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/cv5_summary_mean_std_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/cv5_pooled_test_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/TransMIT_1e-4_256/transmit_paper4_mae_mape_table.csv

FOLDWISE TEST MAE + MAPE:
  fold target                 model  gap_region       mae  mape_pct  n_points_total  n_points_used_mape  n_points_excluded_gt0
    1    MAP TransMIT_WeightedRMSE      <1 min  2.776165  6.898208            2570                2570                      0
    2    MAP TransMIT_WeightedRMSE      <1 min  2.033760  3.743917            2432        